In [ ]:

import os
import random
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split



: 

In [ ]:
SEED = 0          # même graine que 1-Prepocessing : découpage identique
D = 50            # dimension de représentation, IDENTIQUE pour la PCA et l'AE
TEST_SIZE = 0.2   # part de cellules en validation
BATCH_SIZE = 128

In [ ]:
X

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.width", 200)

repertoire = os.getcwd()
DATA_DIR = os.path.dirname(repertoire)

In [ ]:
train = sc.read_h5ad(f"{DATA_DIR}/train.h5ad")
val = sc.read_h5ad(f"{DATA_DIR}/val.h5ad")


In [ ]:
Xt = train.X
Xv = val.X

if sp.issparse(Xt):
    X_train = Xt.toarray()
else:
    X_train = Xt
X_train = np.asarray(X_train, dtype = np.float32)

if sp.issparse(Xv):
    X_val = Xv.toarray()
else:
    X_val = Xv
X_val = np.asarray(X_val, dtype = np.float32)

print(X_train)
print(X_train.shape)
print(X_val.shape)

## Structure du réseaux


In [ ]:
N_EPOCHS = 200
LR = 1e-3



class Autoencodeur(nn.Module):
    def __init__(self, n, d):
        super().__init__()
        self.encodeur = nn.Sequential(
            nn.Linear(n, d),
            nn.Tanh(),
        )
        self.decodeur = nn.Sequential(
            nn.Linear(d, n),
        )

    def forward(self, x):
        return self.decodeur(self.encodeur(x))

    def encode(self, x):
        return self.encodeur(x)


n = X.shape[1]   # nombre de gènes = 18771
d = 64
net = Autoencodeur(n, d)

criterion = nn.MSELoss()
output_fn = nn.Sigmoid()
optimizer = optim.SGD(net.parameters(), lr=0.001)

n = X_train.shape[1]
net = Autoencodeur(n, D)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)

X_train_t = torch.from_numpy(X_train)
X_val_t = torch.from_numpy(X_val)

loader = DataLoader(
    TensorDataset(X_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)


### AE profond


In [ ]:
class AutoencodeurProfond(nn.Module):
    def __init__(self, n, d, couches=(512, 128)):
        super().__init__()
        h1, h2 = couches

        # encodeur : n -> h1 -> h2 -> d
        # on prend Relu pour les couche car plus rapide
        #on garde tanh pour le goulot à la fin
        self.encodeur = nn.Sequential(
            nn.Linear(n, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, d),
            nn.Tanh(),         
        )
        # décodeur : d -> h2 -> h1 -> n
        self.decodeur = nn.Sequential(
            nn.Linear(d, h2),
            nn.ReLU(),
            nn.Linear(h2, h1),
            nn.ReLU(),
            nn.Linear(h1, n),   
        )

    def forward(self, x):
        return self.decodeur(self.encodeur(x))

    def encode(self, x):
        return self.encodeur(x)


net_couche = AutoencodeurProfond(n, D, couches=(512, 128))
optimizer_couche = torch.optim.Adam(net_couche.parameters(), lr=1e-3)
print(sum(p.numel() for p in net.parameters()), "paramètres")




## Entrainement

In [ ]:
def train_session(net, loader, X_val_t, criterion, optimizer, n_epochs, print_every=20):
    hist = {"train": [], "val": []}
    for epoch in range(n_epochs):
        net.train()
        cumul, n_vus = 0.0, 0
        for (batch,) in loader:
            optimizer.zero_grad()
            perte = criterion(net(batch), batch)
            perte.backward()
            optimizer.step()
            cumul += perte.item() * batch.shape[0]
            n_vus += batch.shape[0]

        net.eval()
        with torch.no_grad():
            perte_val = criterion(net(X_val_t), X_val_t).item()

        hist["train"].append(cumul / n_vus)
        hist["val"].append(perte_val)
        if (epoch + 1) % print_every == 0:
            print(f"époque {epoch + 1:3d} | MSE train {hist['train'][-1]:.4f} | val {perte_val:.4f}")

    return hist  

## lancement train AE simple

In [ ]:
print(sum(p.numel() for p in net.parameters()), "paramètres")

# mesure du temps de calcul de l'entraienement 
t0 = time.perf_counter()
historique = train_session(net, loader, X_val_t, criterion, optimizer, N_EPOCHS)
temps_ae = time.perf_counter() - t0

print(f"Temps d'entraînement AE : {temps_ae:.1f} s")

## lancement AE profond

In [ ]:
# mesure du temps d'execution 

t0 = time.perf_counter()
historique2 = train_session(net_couche, loader, X_val_t, criterion, optimizer_couche, N_EPOCHS)
temps_ae = time.perf_counter() - t0

print(f"Temps d'entraînement AE : {temps_ae:.1f} s")

### Valeur initial de la perte
#### Fait une premiere projection dans le reseau pour voir ce qui est reconstruit en sortie

In [ ]:
#projection avec les poids aléatoires
ZPred = net.forward(X_tensor)

#affichage
ZPred[:10,:]

In [ ]:
#valeur de la perte : observé vs. reconstitution
print(criterion(ZPred,X_tensor))


### on peu regarder les poid répartie aléatoirement dans la couche 1 (ici encodeur)

In [ ]:
net.encodeur[0].weight


### on peu regarder s'il y a une constante (biais) deja dans le réseaux


In [ ]:
net.encodeur[0].bias


## Évaluation de l'autoencodeur entraîné

Une fois l'entraînement terminé, on fige le réseau (`net.eval()`) et on l'utilise sans modifier ses poids. Le calcul des gradients est désactivé (`torch.no_grad()`), car on ne fait plus de rétropropagation : cela économise de la mémoire et du temps.

On passe les cellules du train et de la validation dans le modèle de deux façons :

- **Encodage** (`net.encode`) : l'encodeur seul compresse chaque cellule de 2 000 gènes en un vecteur de **50 valeurs**, sa représentation latente (`Z_ae_train`, `Z_ae_val`). C'est ce qui sera visualisé en 2D et utilisé pour le clustering.
- **Reconstruction** (`net`) : l'encodeur puis le décodeur reconstruisent les 2 000 gènes à partir de ces 50 valeurs (`X_ae_train`, `X_ae_val`).

On mesure ensuite l'erreur de reconstruction (MSE) entre les valeurs d'origine et les valeurs reconstruites, sur le train et sur la validation. Ce calcul est fait avec le modèle final sur l'ensemble des cellules, donc il est plus propre que la courbe d'apprentissage, qui est une moyenne calculée pendant l'époque, alors que les poids changent encore. C'est cette MSE qui est comparable à celle de la PCA et à la référence « moyenne par gène ». 

In [ ]:
def evaluer(net):
    net.eval()
    with torch.no_grad():
        Z_train = net.encode(X_train_t).numpy()
        Z_val = net.encode(X_val_t).numpy()
        Xr_train = net(X_train_t).numpy()
        Xr_val = net(X_val_t).numpy()
    return {
        "Z_train": Z_train, "Z_val": Z_val,
        "mse_train": mean_squared_error(X_train, Xr_train),
        "mse_val": mean_squared_error(X_val, Xr_val),
    }

res = evaluer(net)
res_couche = evaluer(net_couche)

print(f"AE simple  - train : {res['mse_train']:.4f} | val : {res['mse_val']:.4f}")
print(f"AE couche - train : {res_couche['mse_train']:.4f} | val : {res_couche['mse_val']:.4f}")

## visualisation de la perte

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historique["train"], label="train")
plt.plot(historique["val"], label="validation")
plt.xlabel("époque")
plt.ylabel("MSE")
plt.title("Courbe d'apprentissage de l'autoencodeur")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historique2["train"], label="train")
plt.plot(historique2["val"], label="validation")
plt.xlabel("époque")
plt.ylabel("MSE")
plt.title("Courbe d'apprentissage de l'autoencodeur")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import os
import numpy as np

os.makedirs("results", exist_ok=True)

np.savez_compressed(
    "results/resultats_AE.npz",

    # AE simple
    Z_train=res["Z_train"],
    Z_val=res["Z_val"],
    mse_train=res["mse_train"],
    mse_val=res["mse_val"],

    # AE profond
    Z_train_profond=res_couche["Z_train"],
    Z_val_profond=res_couche["Z_val"],
    mse_train_profond=res_couche["mse_train"],
    mse_val_profond=res_couche["mse_val"],

    # Historiques d'entraînement
    historique_train=historique["train"],
    historique_val=historique["val"],
    historique2_train=historique2["train"],
    historique2_val=historique2["val"],

    # Identification des cellules
    train_obs_names=train.obs_names.to_numpy(),
    val_obs_names=val.obs_names.to_numpy()
)

print("Résultats AE sauvegardés.")